# 09 - resolution and field of view: what we feed the model today, and what we could

The loader rescales every slice to a fixed millimetres-per-pixel, then centre-crops or pads it to a fixed pixel size. Two numbers set that:

- `mm_per_px` - how many millimetres of real knee each pixel covers. Smaller = finer detail.
- `size` - the output side in pixels.

Their product is the **field of view**: how many millimetres of anatomy are in frame. Today that is `0.40 x 512 = 205 mm`.

The scanners in this dataset typically acquire **512 x 512 pixels over a 160 mm field of view**, i.e. **0.312 mm per pixel**. So on a typical study we currently shrink a native 512 px image down to about 400 px and then pad it back out to 512. We lose real detail and spend about a fifth of every image on padding.

This notebook shows that on real studies, side by side, so the tradeoff is visible rather than argued.

Run one cell at a time.

In [1]:
# Cell 1 - load the local shards and see what we actually have to work with.
# Shards store each series at its NATIVE pixel size, with the DICOM pixel spacing in the
# metadata, and the loader does the rescaling at training time. That is what makes this
# comparison honest: we can replay both settings from the same stored pixels.

import json
from pathlib import Path

import numpy as np
import pandas as pd

if "ROOT" not in globals():
    ROOT = Path.cwd().parent if Path.cwd().name == "data_code" else Path.cwd()
SHARD_DIR = ROOT / "data" / "shards"


def read_shard(path):
    """(meta dict, {series_uid: uint8 array of shape (n_slices, h, w)}) for one shard."""
    z = np.load(path, allow_pickle=True)
    meta = json.loads(bytes(z["meta"]).decode("utf-8"))
    vols = {k: z[k] for k in z.files if k != "meta"}
    return meta, vols


shard_files = sorted(SHARD_DIR.glob("*.npz"))
print(f"{len(shard_files)} shards locally under {SHARD_DIR}")

rows = []
for f in shard_files:
    meta, vols = read_shard(f)
    for uid, info in meta["series"].items():
        if uid not in vols:
            continue
        n, h, w = vols[uid].shape
        sp = (info.get("pixel_spacing") or [None, None])[1]
        rows.append({
            "shard": f.name,
            "study": meta["study"],
            "series": uid,
            "desc": info.get("series_desc", ""),
            "n_slices": n,
            "h": h, "w": w,
            "mm_per_px": sp,
            "fov_mm": None if sp is None else round(sp * w, 1),
        })

S = pd.DataFrame(rows)
print(f"{S.study.nunique()} studies, {len(S)} series\n")
print("native pixel spacing (mm per pixel):")
print(S.mm_per_px.describe().round(3).to_string())
print("\nnative field of view (mm across the image):")
print(S.fov_mm.describe().round(1).to_string())
print("\nWhat to look for: a median spacing near 0.31 and a median field of view near 160")
print("mean the scanner already gave us finer pixels than the 0.40 we resample to, over a")
print("narrower frame than the 205 mm we ask for.")
S.head(10)

5 shards locally under d:\My Projects\knee_abnormality_detection\data\shards
5 studies, 31 series

native pixel spacing (mm per pixel):
count    31.000
mean      0.319
std       0.104
min       0.137
25%       0.250
50%       0.299
75%       0.417
max       0.625

native field of view (mm across the image):
count     31.0
mean     159.1
std       16.1
min      100.0
25%      150.0
50%      160.0
75%      169.0
max      180.0

What to look for: a median spacing near 0.31 and a median field of view near 160
mean the scanner already gave us finer pixels than the 0.40 we resample to, over a
narrower frame than the 205 mm we ask for.


,shard,study,series,desc,n_slices,h,w,mm_per_px,fov_mm
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.12343110195036213483...,DP SPIR CS_SAG,22,512,512,0.330078,169.0
1,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.13821229744997220641...,STIR CS_ AX,18,560,560,0.287712,161.1
2,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.23084836536722595275...,T1_TSE_CS,16,576,576,0.298910,172.2
3,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.40734206102458723096...,DP SPAIR_CS,16,560,560,0.307450,172.2
4,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.75714899997203615784...,DP CS_SAG,22,640,640,0.264062,169.0
5,1.2.826.0.1.3680043.8.498.10009278692606631573...,1.2.826.0.1.3680043.8.498.10009278692606631573...,1.2.826.0.1.3680043.8.498.12689544351656485342...,pd_tse_fs_cor_416_p2_s2,38,416,416,0.432692,180.0
6,1.2.826.0.1.3680043.8.498.10009278692606631573...,1.2.826.0.1.3680043.8.498.10009278692606631573...,1.2.826.0.1.3680043.8.498.26515132645256697953...,pd_tse_fs_tra_416_p2_s2,48,416,416,0.432692,180.0
7,1.2.826.0.1.3680043.8.498.10009278692606631573...,1.2.826.0.1.3680043.8.498.10009278692606631573...,1.2.826.0.1.3680043.8.498.30827030879314217305...,t1_tse_cor_448_p2_s2,38,416,416,0.432692,180.0
8,1.2.826.0.1.3680043.8.498.10009278692606631573...,1.2.826.0.1.3680043.8.498.10009278692606631573...,1.2.826.0.1.3680043.8.498.33836648034724869721...,t2_tse_sag_448_2.5mm,38,416,416,0.432692,180.0
9,1.2.826.0.1.3680043.8.498.10009278692606631573...,1.2.826.0.1.3680043.8.498.10009278692606631573...,1.2.826.0.1.3680043.8.498.93288538765996574291...,pd_tse_fs_sag_416_p2_s2,38,416,416,0.432692,180.0
